# Learning Objectives

* Build a fan-out Pegasus workflow from prebuilt radar nowcasting tools in an Apptainer container.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the per-minute reflectivity images and storm outlines produced by a run.

# CASA Nowcast Workflow

This workflow post-processes CASA (Collaborative Adaptive Sensing of the Atmosphere) 30-minute radar
nowcasts. Each forecast file is split into 31 per-minute predicted-reflectivity fields; every minute is then
thresholded into a storm GeoJSON and rendered to a PNG in parallel. See `README.md` in this repository for the
full description.

**Workflow Jobs (per forecast file):**
1. `NowcastToWDSS2` - split the forecast into 31 per-minute reflectivity files
2. `mrtV2` - threshold each minute into a storm GeoJSON (31 jobs)
3. `merged_netcdf2png` - render each minute into a PNG (31 jobs)

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Sample Input Data

Sample forecast files ship in `input/` (`MERGE_DARTS_YYYYMMDDHHMM.nc`, one per minute from 2019-03-25
05:00 UTC), with the `mrtV2` threshold config and the reflectivity color scale. This notebook uses the first
one; pass more to `FILES` below to combine them into a single workflow.

In [ ]:
!ls input/ | head -5; ls input/ | wc -l

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/CASA_Nowcast_Container.def`) converted from the published
`pegasus/casa-nowcast` Docker image, which ships the tools prebuilt. Pegasus stages the `.sif` to the workers, so jobs need
no registry access. Building it can take a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/CASA_Nowcast_Container.sif ]; then
    apptainer build Apptainer/CASA_Nowcast_Container.sif Apptainer/CASA_Nowcast_Container.def
else
    echo "Container already built: Apptainer/CASA_Nowcast_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `CASAWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs, embedded in
`workflow.yml`) and writes it with `pegasus.properties`. Nothing about the workflow is defined here: change the
pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` adds a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import CASAWorkflow, resolve_forecast_file

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

FORECAST_FILES = [resolve_forecast_file(f) for f in ["MERGE_DARTS_201903250500.nc"]]
print(f"Input files: {[Path(f).name for f in FORECAST_FILES]}")

workflow = CASAWorkflow(outdir=str(Path(".").resolve()), forecast_files=FORECAST_FILES)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating CASA Nowcast workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nCASA Nowcast Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph fans out from one `NowcastToWDSS2` job into 31 parallel `mrtV2` / `merged_netcdf2png` pairs, one per forecast minute.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (with `HOSTED_SITE_CATALOG` set, Pegasus's
default `wf-output/` instead):

| Output | Description |
|--------|-------------|
| `PredictedReflectivity_{N}min_{YYYYMMDD}-{HHMMSS}.nc` | Predicted reflectivity N minutes ahead (N = 0..30) |
| `mrt_STORM_CASA_{N}_{YYYYMMDD}-{HHMMSS}.geojson` | Storm outlines above the reflectivity threshold |
| `PredictedReflectivity_{N}min_{YYYYMMDD}-{HHMMSS}.png` | Rendered reflectivity field |

In [ ]:
!ls -lt output/

The predicted reflectivity 15 minutes ahead, and the storm outlines extracted from it:

In [ ]:
Image(filename="output/PredictedReflectivity_15min_20190325-050000.png")

In [ ]:
!head -c 600 output/mrt_STORM_CASA_15_20190325-050000.geojson